In [187]:
import os 
import sys
import json
import pandas as pd
import numpy as np
from scipy.stats import rankdata
from scipy.stats import gamma
from scipy.optimize import minimize
import matplotlib.pyplot as plt
from scipy.stats import trim_mean
import seaborn as sns
import sys
import ast
sys.path.append('./models/')
from useful_functions import df_to_dict, concat_dico, get_classement, sort_list
from evaluate_model import WIS


In [208]:
models1Dnames=['Moving Average','ARIMA', 'Exponential', 'Linear Regression', 'Bayesian Regression','SIRH1', 'SIRH2', 'SIRH3', 'SIRH4']
models3Dnames=[ 'VAR', 'Exponential Multi', 'SIRH Multi1', 'SIRH Multi2','SEIR Mob']
ens_names=['EnsAvg','EnsMedian','EnsRegr','EnsRMSE','EnsWIS','EnsRank']
reach=7
trim=0.05
list_of_models= models1Dnames+models3Dnames
print(list_of_models)

['Moving Average', 'ARIMA', 'Exponential', 'Linear Regression', 'Bayesian Regression', 'SIRH1', 'SIRH2', 'SIRH3', 'SIRH4', 'VAR', 'Exponential Multi', 'SIRH Multi1', 'SIRH Multi2', 'SEIR Mob']


In [217]:
def one_sided_trim_mean(x, proportiontocut=0.05):
    """
    Trim only the upper tail (largest values).
    proportiontocut = fraction to cut from the top (e.g. 0.05 = top 5%).
    """
    x = np.sort(x)
    n = len(x)
    k = int(proportiontocut * n)
    if k == 0:
        return np.mean(x)
    return np.mean(x[:-k])

In [218]:
def classify_bis(point, r_effs):  # Classification based on transmission dynamics

    if r_effs[point] < 0.5:
        return 'minimal transmission'
    elif r_effs[point] < 0.8:
        return 'low transmission'
    elif r_effs[point] < 1.2:
        return 'stable'
    elif r_effs[point] < 3:
        return 'high transmission'
    else:
        return 'very high transmission'

In [219]:
def est_R(incidence):
    # Given parameters
    mean = 4
    std_dev = 3
    
    # Calculate shape (k) and scale (theta) parameters
    shape = (mean / std_dev) ** 2  # k (shape parameter)
    scale = std_dev ** 2 / mean    # theta (scale parameter)
    
    # Generate integer values to evaluate the PDF
    x_values_int = np.arange(0, 21)  # Integer values from 0 to 20
    
    # Compute the PDF at integer values
    GT = gamma.pdf(x_values_int, a=shape, scale=scale)
    gen_time_pmf = GT/np.sum(GT)
    max_days=20
    incidence=np.convolve(incidence, np.ones(7), 'valid') / 7
    R_t = []
    for t in range(len(incidence)):
        if t == 0:  # Skip the first day (no prior data)
            R_t.append(np.nan)
            continue
        
        # Calculate the sum of weighted past incidences
        weighted_sum = 0
        for s in range(1, min(t + 1, max_days)):
            weighted_sum += gen_time_pmf[s] * incidence[t - s]
        
        if weighted_sum > 0:
            R_t.append(incidence[t] / weighted_sum)
        else:
            R_t.append(np.nan)

    return np.array(R_t)

In [220]:
def R_case(Rt):
    if Rt < 0.5:
        return 5
    elif Rt < 0.8:
        return 4
    elif Rt < 1.2:
        return 3
    elif Rt < 3:
        return 2
    else:
        return 1

In [221]:
def safe_eval(x):
    replacement_value = [(0.0, 0.0)] * 11
    try:
        if isinstance(x, str):
            # Convert the string representation to an actual Python object
            evaluated = ast.literal_eval(x)

            # Check for (-inf, inf) tuples and replace with a default value (e.g., (0, 0))
            evaluated = [(0, 0) if (tup == (-np.inf, np.inf)) else tup for tup in evaluated]

            return evaluated
        else:
            return x
    except (ValueError, SyntaxError):
        return replacement_value  # Return None for invalid entries

In [222]:
def get_quantiles(w,row):
    q = [(0.0, 0.0)] * 11
    for i in range(len(list_of_models)):
        q = q + w[i]*np.array(row.iloc[i])
    return [tuple(row) for row in q]

In [223]:
def get_median_quantiles(row):
    q = np.zeros((11,2,len(list_of_models)))
    for i in range(len(list_of_models)):
        q[:,:,i] = np.array(row.iloc[i])
    mq=np.median(q,axis=2)
    return [tuple(row) for row in mq]

In [195]:
#Define quantiles and weights
alphas=np.array([0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])
weights=np.concatenate((np.array([0.5]), alphas * 0.5))
    
#Create training and test data
pointpred_train=pd.DataFrame(columns =[name for name in  list_of_models + ['Real values']])
pointpred_test=pd.DataFrame(columns =[name for name in  list_of_models + ['Real values']])
quant_train=pd.DataFrame(columns =[name for name in  list_of_models])
quant_test=pd.DataFrame(columns =[name for name in  list_of_models])

#names = [name for name in os.listdir('./results/predictions_of_the_models/') if 's_'+str(reach) in name if 'predictions' in name]
names=['belgium','sweden','france','czechia','US'] #
for name in names: 
    prediction=pd.read_csv('./results/predictions_of_the_models/countries/'+'predictions_'+str(reach)+'_days_on_pandemic_test_'+name+'.csv')
    prediction.drop(['Unnamed: 0'], axis=1, inplace=True)
    prediction.index=[20 * i for i in range(1, 15)]
    quants=pd.read_csv('./results/predictions_of_the_models/countries/'+'quantiles_'+str(reach)+'_days_on_pandemic_test_'+name+'.csv')
    quants=quants.applymap(safe_eval)
    quants.drop(['Unnamed: 0'], axis=1, inplace=True)
    quants.index=[20 * i for i in range(1, 15)]
    quant_test=pd.concat([quant_test, quants])

/var/folders/vq/kbhqcbz52js6b8nz5zvnryc1fy9f2x/T/ipykernel_17615/3242521169.py:18: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  quants=quants.applymap(safe_eval)
/var/folders/vq/kbhqcbz52js6b8nz5zvnryc1fy9f2x/T/ipykernel_17615/3242521169.py:18: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  quants=quants.applymap(safe_eval)
/var/folders/vq/kbhqcbz52js6b8nz5zvnryc1fy9f2x/T/ipykernel_17615/3242521169.py:18: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  quants=quants.applymap(safe_eval)
/var/folders/vq/kbhqcbz52js6b8nz5zvnryc1fy9f2x/T/ipykernel_17615/3242521169.py:18: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  quants=quants.applymap(safe_eval)
/var/folders/vq/kbhqcbz52js6b8nz5zvnryc1fy9f2x/T/ipykernel_17615/3242521169.py:18: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  quants=quants.applymap(safe_e

In [196]:
#Collect all ensemble weights
ens_weights=pd.read_csv('./ensemble_weights.csv')
ens_weights.drop(['Unnamed: 0'], axis=1, inplace=True)
#print(ens_weights)
EnsW1=np.array(ens_weights.loc[0,:])
EnsW2=np.array(ens_weights.loc[1,:])
EnsW3=np.array(ens_weights.loc[2,:])
EnsW4=np.array(ens_weights.loc[3,:])
EnsW5=np.array(ens_weights.loc[4,:])

Inter3=339.85

#print(EnsW1,EnsW2,EnsW3,EnsW4,EnsW5,EnsW6)
#print(model_ranksp)

In [230]:
###RMSE 7 days####
import json
import numpy as np
import pandas as pd

reach=7
loss='RMSE'
model_ranks=pd.read_csv('./ranks_for_ensemble_'+loss+'_'+str(reach)+'.csv')

list_of_models_ens = models1Dnames + models3Dnames + ens_names
n_models = len(list_of_models_ens)

rank_counts = np.zeros((n_models, n_models))   # [model, rank]
rmse_sums   = np.zeros(n_models)
mape_sums   = np.zeros(n_models)
mape=[]
k = 0  # counter of points

for name in names:
    eval_name = f"evaluation_with_{loss}_of_1D_models_on_pandemic_test_{name}_and_reach_={reach}.json"
    with open('./results/global_evaluation/countries/' + eval_name, 'r') as f:
        dico1 = json.load(f)
    with open('./results/global_evaluation/countries/' + eval_name.replace('1D', '3D'), 'r') as f:
        dico2 = json.load(f)
    dicoresults = concat_dico(dico1, dico2)

    df = pd.read_csv(f'./all_pandemics/countries/pandemic_test_{name}.csv')
    df.index = ['n_hospitalized', 'n_infectious', 'mobility', 'r_eff']    
    df.drop(['Unnamed: 0'], axis=1, inplace=True)
    n_hospitalized = df.loc['n_hospitalized'].to_numpy(dtype=float)
    r_eff=est_R(df.loc['n_infectious'].to_numpy())
    
    prediction = pd.read_csv(f'./results/predictions_of_the_models/countries/predictions_{reach}_days_on_pandemic_test_{name}.csv')
    real_values = prediction['Real values'].to_numpy(dtype=float)
    prediction.drop(['Unnamed: 0','Real values'], axis=1, inplace=True)
    prediction.index = [20*i for i in range(1,15)]

    for i, pt in enumerate(prediction.index):
        if pt >= len(n_hospitalized): 
            continue
        if n_hospitalized[pt] < 10:
            continue

        # base-model errors from eval dict
        perfs = dicoresults[str([pt])] if str([pt]) in dicoresults else dicoresults[str(pt)]
        perfs = list(map(float, perfs))

        # ensemble predictions
        row = prediction.iloc[i].to_numpy(dtype=float)
        Rpoint=R_case(r_eff[pt])
        #print(pt,name,r_eff[pt],Rpoint)
                    
        pred_ens1 = np.mean(row)
        pred_ens2 = np.median(row)
        pred_ens3 = np.dot(row, EnsW3) + Inter3
        pred_ens4 = np.dot(row, EnsW4)
        pred_ens5 = np.dot(row, EnsW5)
        pred_ens6=np.dot(row,model_ranks.iloc[Rpoint,1:])
        #print(model_ranks.iloc[Rpoint,1:].to_numpy())
        #pred_ens6 = np.dot(row, EnsW6)
        
        
        ens_pred = np.array([pred_ens1, pred_ens2, pred_ens3, pred_ens4, pred_ens5, pred_ens6])
        real_val = real_values[i]
        ens_perf = np.sqrt((ens_pred - real_val)**2)

        #print(ens_perf[5],ens_perf[1])
        perfs = np.array(perfs + ens_perf.tolist())
        rankings = get_classement(perfs)
        mape.append(perfs/real_val)
        #print(perfs/real_val)
        
        for m, r in enumerate(rankings):
            rank_counts[m, r] += 1
            rmse_sums[m] += perfs[m]
            #mape_sums[m] += mape[m]
        k += 1

# ---- expected rank for each model ----
ranks = np.arange(n_models)
expected_ranks = (rank_counts * ranks).sum(axis=1) / rank_counts.sum(axis=1) /n_models

MAPE = np.vstack(mape)
tmape = np.apply_along_axis(
    lambda col: one_sided_trim_mean(col, proportiontocut=trim),  # trim 10% each tail
    axis=0,
    arr=MAPE
)

expected_ranks_rmse_7 = expected_ranks
mape_7 = tmape

df_expected_ranks = pd.DataFrame({
    "model": list_of_models_ens,
    "expected_rank": expected_ranks,
    "rmse_mean": rmse_sums / k,
    "mape_mean": tmape
}).sort_values("expected_rank").reset_index(drop=True)

#print(df_expected_ranks)

In [229]:
###RMSE 14 days####
import json
import numpy as np
import pandas as pd

reach=14
loss='RMSE'
model_ranks=pd.read_csv('./ranks_for_ensemble_'+loss+'_'+str(reach)+'.csv')

list_of_models_ens = models1Dnames + models3Dnames + ens_names
n_models = len(list_of_models_ens)

rank_counts = np.zeros((n_models, n_models))   # [model, rank]
rmse_sums   = np.zeros(n_models)
mape_sums   = np.zeros(n_models)
mape=[]
k = 0  # counter of points

for name in names:
    eval_name = f"evaluation_with_{loss}_of_1D_models_on_pandemic_test_{name}_and_reach_={reach}.json"
    with open('./results/global_evaluation/countries/' + eval_name, 'r') as f:
        dico1 = json.load(f)
    with open('./results/global_evaluation/countries/' + eval_name.replace('1D', '3D'), 'r') as f:
        dico2 = json.load(f)
    dicoresults = concat_dico(dico1, dico2)

    df = pd.read_csv(f'./all_pandemics/countries/pandemic_test_{name}.csv')
    df.index = ['n_hospitalized', 'n_infectious', 'mobility', 'r_eff']    
    df.drop(['Unnamed: 0'], axis=1, inplace=True)
    n_hospitalized = df.loc['n_hospitalized'].to_numpy(dtype=float)
    r_eff=est_R(df.loc['n_infectious'].to_numpy())
    
    prediction = pd.read_csv(f'./results/predictions_of_the_models/countries/predictions_{reach}_days_on_pandemic_test_{name}.csv')
    real_values = prediction['Real values'].to_numpy(dtype=float)
    prediction.drop(['Unnamed: 0','Real values'], axis=1, inplace=True)
    prediction.index = [20*i for i in range(1,15)]

    for i, pt in enumerate(prediction.index):
        if pt >= len(n_hospitalized): 
            continue
        if n_hospitalized[pt] < 10:
            continue

        # base-model errors from eval dict
        perfs = dicoresults[str([pt])] if str([pt]) in dicoresults else dicoresults[str(pt)]
        perfs = list(map(float, perfs))

        # ensemble predictions
        row = prediction.iloc[i].to_numpy(dtype=float)
        Rpoint=R_case(r_eff[pt])
        #print(pt,name,r_eff[pt],Rpoint)
                    
        pred_ens1 = np.mean(row)
        pred_ens2 = np.median(row)
        pred_ens3 = np.dot(row, EnsW3) + Inter3
        pred_ens4 = np.dot(row, EnsW4)
        pred_ens5 = np.dot(row, EnsW5)
        pred_ens6=np.dot(row,model_ranks.iloc[Rpoint,1:])
        #print(model_ranks.iloc[Rpoint,1:].to_numpy())
        #pred_ens6 = np.dot(row, EnsW6)
        
        
        ens_pred = np.array([pred_ens1, pred_ens2, pred_ens3, pred_ens4, pred_ens5, pred_ens6])
        real_val = real_values[i]
        ens_perf = np.sqrt((ens_pred - real_val)**2)

        #print(ens_perf[5],ens_perf[1])
        perfs = np.array(perfs + ens_perf.tolist())
        rankings = get_classement(perfs)
        mape.append(perfs/real_val)
        #print(perfs/real_val)
        
        for m, r in enumerate(rankings):
            rank_counts[m, r] += 1
            rmse_sums[m] += perfs[m]
            #mape_sums[m] += mape[m]
        k += 1

# ---- expected rank for each model ----
ranks = np.arange(n_models)
expected_ranks = (rank_counts * ranks).sum(axis=1) / rank_counts.sum(axis=1) /n_models

MAPE = np.vstack(mape)
tmape = np.apply_along_axis(
    lambda col: one_sided_trim_mean(col, proportiontocut=trim),  # trim 10% each tail
    axis=0,
    arr=MAPE
)

expected_ranks_rmse_14 = expected_ranks
mape_14 = tmape

df_expected_ranks = pd.DataFrame({
    "model": list_of_models_ens,
    "expected_rank": expected_ranks,
    "rmse_mean": rmse_sums / k,
    "mape_mean": tmape
}).sort_values("expected_rank").reset_index(drop=True)

#print(df_expected_ranks)

In [215]:
###WIS 7 days####
import json
import numpy as np
import pandas as pd

reach=7
loss='WIS'
model_ranks=pd.read_csv('./ranks_for_ensemble_'+loss+'_'+str(reach)+'.csv')

list_of_models_ens = models1Dnames + models3Dnames + ens_names
n_models = len(list_of_models_ens)

rank_counts = np.zeros((n_models, n_models))   # [model, rank]
rmse_sums   = np.zeros(n_models)
k = 0  # counter of points
l = 0
for name_i,name in enumerate(names):
    eval_name = f"evaluation_with_{loss}_of_1D_models_on_pandemic_test_{name}_and_reach_={reach}.json"
    with open('./results/global_evaluation/countries/' + eval_name, 'r') as f:
        dico1 = json.load(f)
    with open('./results/global_evaluation/countries/' + eval_name.replace('1D', '3D'), 'r') as f:
        dico2 = json.load(f)
    dicoresults = concat_dico(dico1, dico2)

    df = pd.read_csv(f'./all_pandemics/countries/pandemic_test_{name}.csv')
    df.index = ['n_hospitalized', 'n_infectious', 'mobility', 'r_eff']    
    df.drop(['Unnamed: 0'], axis=1, inplace=True)
    n_hospitalized = df.loc['n_hospitalized'].to_numpy(dtype=float)
    r_eff=est_R(df.loc['n_infectious'].to_numpy())
    
    prediction = pd.read_csv(f'./results/predictions_of_the_models/countries/predictions_{reach}_days_on_pandemic_test_{name}.csv')
    real_values = prediction['Real values'].to_numpy(dtype=float)
    prediction.drop(['Unnamed: 0','Real values'], axis=1, inplace=True)
    prediction.index = [20*i for i in range(1,15)]

    for i, pt in enumerate(prediction.index):
        if pt >= len(n_hospitalized): 
            continue
        if n_hospitalized[pt] < 10:
            continue

        # base-model errors from eval dict
        perfs = dicoresults[str([pt])] if str([pt]) in dicoresults else dicoresults[str(pt)]
        perfs = list(map(float, perfs))

        # ensemble predictions
        row = prediction.iloc[i].to_numpy(dtype=float)
        Rpoint=R_case(r_eff[pt])
        #print(pt,name,r_eff[pt],Rpoint)
                    
        pred_ens1 = np.mean(row)
        pred_ens2 = np.median(row)
        pred_ens3 = np.dot(row, EnsW3) + Inter3
        pred_ens4 = np.dot(row, EnsW4)
        pred_ens5 = np.dot(row, EnsW5)
        EnsW6 = model_ranks.iloc[Rpoint,1:]
        pred_ens6=np.dot(row,model_ranks.iloc[Rpoint,1:])
        #print(model_ranks.iloc[Rpoint,1:].to_numpy())
        #pred_ens6 = np.dot(row, EnsW6)
        
        
        ens_pred = np.array([pred_ens1, pred_ens2, pred_ens3, pred_ens4, pred_ens5, pred_ens6])
        real_value = real_values[i]

        ens_pred=np.array([pred_ens1,pred_ens2,pred_ens3,pred_ens4,pred_ens5,pred_ens6])

        inds=np.arange(14*name_i,14*name_i+14)
        point_quants=quant_test.iloc[inds]
        quant_ens1=get_quantiles(EnsW1,point_quants.iloc[i])
        quant_ens2=get_median_quantiles(point_quants.iloc[i])
        quant_ens3=get_quantiles(EnsW3,point_quants.iloc[i])
        quant_ens3 = [(x + Inter3, y + Inter3) for x, y in quant_ens3]
        quant_ens4=get_quantiles(EnsW4,point_quants.iloc[i])
        quant_ens5=get_quantiles(EnsW5,point_quants.iloc[i])
        quant_ens6=get_quantiles(EnsW6,point_quants.iloc[i])

        wis_ens1=WIS(real_value,quant_ens1,pred_ens1,alphas,weights)
        wis_ens2=WIS(real_value,quant_ens2,pred_ens2,alphas,weights)
        wis_ens3=WIS(real_value,quant_ens3,pred_ens3,alphas,weights)
        wis_ens4=WIS(real_value,quant_ens4,pred_ens4,alphas,weights)
        wis_ens5=WIS(real_value,quant_ens5,pred_ens5,alphas,weights)
        wis_ens6=WIS(real_value,quant_ens6,pred_ens6,alphas,weights)
        
        ens_perf = np.array([wis_ens1,wis_ens2,wis_ens3,wis_ens4,wis_ens5,wis_ens6])                         
        #perfs=dicoresults[str(point)]
        #perfs.extend(ens_perf)
        
        #print(ens_perf[5],ens_perf[1])
        if ens_perf[5]<ens_perf[1]:
            l += 1
        perfs = np.array(perfs + ens_perf.tolist())
        rankings = get_classement(perfs)
        for m, r in enumerate(rankings):
            rank_counts[m, r] += 1
            rmse_sums[m] += perfs[m]
        k += 1

# ---- expected rank for each model ----
ranks = np.arange(n_models)
expected_ranks = (rank_counts * ranks).sum(axis=1) / rank_counts.sum(axis=1) /n_models

expected_ranks_wis_7 = expected_ranks

df_expected_ranks = pd.DataFrame({
    "model": list_of_models_ens,
    "expected_rank": expected_ranks,
    "wis_mean": rmse_sums / k
}).sort_values("expected_rank").reset_index(drop=True)

#print(df_expected_ranks)


/var/folders/vq/kbhqcbz52js6b8nz5zvnryc1fy9f2x/T/ipykernel_17615/1508672954.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  q = q + w[i]*np.array(row.iloc[i])
/var/folders/vq/kbhqcbz52js6b8nz5zvnryc1fy9f2x/T/ipykernel_17615/1508672954.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  q = q + w[i]*np.array(row.iloc[i])
/var/folders/vq/kbhqcbz52js6b8nz5zvnryc1fy9f2x/T/ipykernel_17615/1508672954.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use 

In [216]:
#Fraction of cases Rank outperforms Median
print(l/k)

0.4


In [213]:
###WIS 14 days####
import json
import numpy as np
import pandas as pd

reach=14
loss='WIS'
model_ranks=pd.read_csv('./ranks_for_ensemble_'+loss+'_'+str(reach)+'.csv')

list_of_models_ens = models1Dnames + models3Dnames + ens_names
n_models = len(list_of_models_ens)

rank_counts = np.zeros((n_models, n_models))   # [model, rank]
rmse_sums   = np.zeros(n_models)
k = 0  # counter of points
l = 0
for name_i,name in enumerate(names):
    eval_name = f"evaluation_with_{loss}_of_1D_models_on_pandemic_test_{name}_and_reach_={reach}.json"
    with open('./results/global_evaluation/countries/' + eval_name, 'r') as f:
        dico1 = json.load(f)
    with open('./results/global_evaluation/countries/' + eval_name.replace('1D', '3D'), 'r') as f:
        dico2 = json.load(f)
    dicoresults = concat_dico(dico1, dico2)

    df = pd.read_csv(f'./all_pandemics/countries/pandemic_test_{name}.csv')
    df.index = ['n_hospitalized', 'n_infectious', 'mobility', 'r_eff']    
    df.drop(['Unnamed: 0'], axis=1, inplace=True)
    n_hospitalized = df.loc['n_hospitalized'].to_numpy(dtype=float)
    r_eff=est_R(df.loc['n_infectious'].to_numpy())
    
    prediction = pd.read_csv(f'./results/predictions_of_the_models/countries/predictions_{reach}_days_on_pandemic_test_{name}.csv')
    real_values = prediction['Real values'].to_numpy(dtype=float)
    prediction.drop(['Unnamed: 0','Real values'], axis=1, inplace=True)
    prediction.index = [20*i for i in range(1,15)]

    for i, pt in enumerate(prediction.index):
        if pt >= len(n_hospitalized): 
            continue
        if n_hospitalized[pt] < 10:
            continue

        # base-model errors from eval dict
        perfs = dicoresults[str([pt])] if str([pt]) in dicoresults else dicoresults[str(pt)]
        perfs = list(map(float, perfs))

        # ensemble predictions
        row = prediction.iloc[i].to_numpy(dtype=float)
        Rpoint=R_case(r_eff[pt])
        #print(pt,name,r_eff[pt],Rpoint)
                    
        pred_ens1 = np.mean(row)
        pred_ens2 = np.median(row)
        pred_ens3 = np.dot(row, EnsW3) + Inter3
        pred_ens4 = np.dot(row, EnsW4)
        pred_ens5 = np.dot(row, EnsW5)
        EnsW6 = model_ranks.iloc[Rpoint,1:]
        pred_ens6=np.dot(row,model_ranks.iloc[Rpoint,1:])
        #print(model_ranks.iloc[Rpoint,1:].to_numpy())
        #pred_ens6 = np.dot(row, EnsW6)
        
        
        ens_pred = np.array([pred_ens1, pred_ens2, pred_ens3, pred_ens4, pred_ens5, pred_ens6])
        real_value = real_values[i]

        ens_pred=np.array([pred_ens1,pred_ens2,pred_ens3,pred_ens4,pred_ens5,pred_ens6])

        inds=np.arange(14*name_i,14*name_i+14)
        point_quants=quant_test.iloc[inds]
        quant_ens1=get_quantiles(EnsW1,point_quants.iloc[i])
        quant_ens2=get_median_quantiles(point_quants.iloc[i])
        quant_ens3=get_quantiles(EnsW3,point_quants.iloc[i])
        quant_ens3 = [(x + Inter3, y + Inter3) for x, y in quant_ens3]
        quant_ens4=get_quantiles(EnsW4,point_quants.iloc[i])
        quant_ens5=get_quantiles(EnsW5,point_quants.iloc[i])
        quant_ens6=get_quantiles(EnsW6,point_quants.iloc[i])

        wis_ens1=WIS(real_value,quant_ens1,pred_ens1,alphas,weights)
        wis_ens2=WIS(real_value,quant_ens2,pred_ens2,alphas,weights)
        wis_ens3=WIS(real_value,quant_ens3,pred_ens3,alphas,weights)
        wis_ens4=WIS(real_value,quant_ens4,pred_ens4,alphas,weights)
        wis_ens5=WIS(real_value,quant_ens5,pred_ens5,alphas,weights)
        wis_ens6=WIS(real_value,quant_ens6,pred_ens6,alphas,weights)
        
        ens_perf = np.array([wis_ens1,wis_ens2,wis_ens3,wis_ens4,wis_ens5,wis_ens6])                         
        #perfs=dicoresults[str(point)]
        #perfs.extend(ens_perf)
        
        #print(ens_perf[5],ens_perf[1])
        if ens_perf[5]<ens_perf[1]:
            l += 1
        perfs = np.array(perfs + ens_perf.tolist())
        rankings = get_classement(perfs)
        for m, r in enumerate(rankings):
            rank_counts[m, r] += 1
            rmse_sums[m] += perfs[m]
        k += 1

# ---- expected rank for each model ----
ranks = np.arange(n_models)
expected_ranks = (rank_counts * ranks).sum(axis=1) / rank_counts.sum(axis=1) /n_models

expected_ranks_wis_14 = expected_ranks

df_expected_ranks = pd.DataFrame({
    "model": list_of_models_ens,
    "expected_rank": expected_ranks,
    "wis_mean": rmse_sums / k
}).sort_values("expected_rank").reset_index(drop=True)

#print(df_expected_ranks)


/var/folders/vq/kbhqcbz52js6b8nz5zvnryc1fy9f2x/T/ipykernel_17615/1508672954.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  q = q + w[i]*np.array(row.iloc[i])
/var/folders/vq/kbhqcbz52js6b8nz5zvnryc1fy9f2x/T/ipykernel_17615/1508672954.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  q = q + w[i]*np.array(row.iloc[i])
/var/folders/vq/kbhqcbz52js6b8nz5zvnryc1fy9f2x/T/ipykernel_17615/1508672954.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use 

In [214]:
#Fraction of cases Rank outperforms Median
print(l/k)

0.2714285714285714


In [205]:
df_expected_ranks = pd.DataFrame({
    "Model": list_of_models_ens,
    "RMSE (7 days)": expected_ranks_rmse_7,
    "WIS (7 days)": expected_ranks_wis_7,
    "RMSE (14 days)": expected_ranks_rmse_14,
    "WIS (14 days)": expected_ranks_wis_14
})#.sort_values("Model").reset_index(drop=True)
print(df_expected_ranks)

latex_table = df_expected_ranks.to_latex(index=False, float_format="%.2f")


# Save the LaTeX table to a file for future use

output_path = "./country_rank_table.tex"

with open(output_path, "w") as f:

    f.write(latex_table)

                  Model  RMSE (7 days)  WIS (7 days)  RMSE (14 days)  \
0        Moving Average       0.595714      0.507857        0.509286   
1                 ARIMA       0.434286      0.389286        0.465000   
2           Exponential       0.430000      0.700714        0.403571   
3     Linear Regression       0.403571      0.402857        0.427857   
4   Bayesian Regression       0.384286      0.350000        0.450000   
5                 SIRH1       0.664286      0.596429        0.657143   
6                 SIRH2       0.461429      0.438571        0.440000   
7                 SIRH3       0.551429      0.508571        0.547857   
8                 SIRH4       0.489286      0.575714        0.490000   
9                   VAR       0.489286      0.441429        0.475000   
10    Exponential Multi       0.406429      0.496429        0.375714   
11          SIRH Multi1       0.495714      0.492143        0.490000   
12          SIRH Multi2       0.695000      0.712857        0.70

In [233]:
data = {

    "Model": list_of_models_ens,

    "MAPE (7 days)": mape_7,

    "MAPE (14 days)": mape_14

}

df = pd.DataFrame(data)

#df.iloc[:, 1:] = df.iloc[:, 1:].round(2)+1 #best rank should be 1

# Convert the DataFrame to LaTeX table format

latex_table = df.to_latex(index=False, float_format="%.2f")



# Save the LaTeX table to a file for future use

output_path = "./country_mape_table.tex"

with open(output_path, "w") as f:

    f.write(latex_table)


In [234]:
print(df)

                  Model  MAPE (7 days)  MAPE (14 days)
0        Moving Average       0.266798        0.401183
1                 ARIMA       0.225457        0.499472
2           Exponential       0.226379        0.516318
3     Linear Regression       0.224262        0.946493
4   Bayesian Regression       0.167783        0.531558
5                 SIRH1       0.284882        0.499618
6                 SIRH2       0.191037        0.328968
7                 SIRH3       0.262934        0.548591
8                 SIRH4       0.222117        0.502359
9                   VAR       0.232360        0.464455
10    Exponential Multi       0.197656        0.414716
11          SIRH Multi1       0.294044        0.750980
12          SIRH Multi2       0.969287        3.586313
13             SEIR Mob       0.214901        0.463135
14               EnsAvg       0.232708        1.520395
15            EnsMedian       0.151096        0.281437
16              EnsRegr       0.695655        1.334824
17        